In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('work/flights-larger.csv').getOrCreate()
spark

In [2]:
#Read data from .csv file
flights=spark.read.csv('flights-larger.csv',sep=',',header=True,inferSchema=True,nullValue='NA')

In [3]:
# Get number of records
print("The data contrain %d records."%flights.count())

The data contrain 275000 records.


In [4]:
# View the first five records
flights.show(5)

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
| 10| 10|  1|     OO|  5836|ORD| 157|  8.18|      51|   27|
|  1|  4|  1|     OO|  5866|ORD| 466|  15.5|     102| NULL|
| 11| 22|  1|     OO|  6016|ORD| 738|  7.17|     127|  -19|
|  2| 14|  5|     B6|   199|JFK|2248| 21.17|     365|   60|
|  5| 25|  3|     WN|  1675|SJC| 386| 12.92|      85|   22|
+---+---+---+-------+------+---+----+------+--------+-----+
only showing top 5 rows



In [5]:
# Check column data types
print(flights.printSchema())
print(flights.dtypes)

root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- mile: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)

None
[('mon', 'int'), ('dom', 'int'), ('dow', 'int'), ('carrier', 'string'), ('flight', 'int'), ('org', 'string'), ('mile', 'int'), ('depart', 'double'), ('duration', 'int'), ('delay', 'int')]


In [6]:
flights_drop_column=flights.drop('flight')

#Number of records with missing 'delay' values
flights_drop_column.filter('delay IS NULL').count()

16711

In [7]:
 #Remove records with missing 'delay' values
flights_valid_delay=flights_drop_column.filter('delay IS NOT NULL')

In [8]:
# Remove records with missing values in any column and get the number of remaining rows
flights_none_missing=flights_valid_delay.dropna()
print(flights_none_missing.count())

258289


In [9]:
from pyspark.sql.functions import round

# Convert 'mile' to 'km' and drop 'mile' column
flights_km=flights_none_missing.withColumn('km',round(flights_none_missing.mile*1.6093,0)).drop('mile')

# Create 'label' column indicating whether flight delayed (1) or not(0)
flights_km=flights_km.withColumn('label',(flights_km.delay >=15).cast('integer'))

# Check first five records
flights_km.show(5)

+---+---+---+-------+---+------+--------+-----+------+-----+
|mon|dom|dow|carrier|org|depart|duration|delay|    km|label|
+---+---+---+-------+---+------+--------+-----+------+-----+
| 10| 10|  1|     OO|ORD|  8.18|      51|   27| 253.0|    1|
| 11| 22|  1|     OO|ORD|  7.17|     127|  -19|1188.0|    0|
|  2| 14|  5|     B6|JFK| 21.17|     365|   60|3618.0|    1|
|  5| 25|  3|     WN|SJC| 12.92|      85|   22| 621.0|    1|
|  3| 28|  1|     B6|LGA| 13.33|     182|   70|1732.0|    1|
+---+---+---+-------+---+------+--------+-----+------+-----+
only showing top 5 rows



In [10]:
from pyspark.ml.feature import StringIndexer

# Create an indexer
indexer=StringIndexer(inputCol='carrier',outputCol='carrier_idx')

# Indexer identifies categories in the data
indexer_model=indexer.fit(flights_km)

# Indexer creates a new column with numeric index values
flights_indexed=indexer_model.transform(flights_km)

In [11]:
# Repeat the process for the other categorical feature
flights_indexed = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights_indexed).transform(flights_indexed)

In [12]:
from pyspark.ml.feature import VectorAssembler
# Create an assembler object
assembler=VectorAssembler(inputCols=['mon','dom','dow','carrier_idx','org_idx','km','depart','duration'],outputCol='features')

In [16]:
# Consolidate predictor columns
flights_assembled = assembler.transform(flights_indexed)
# Check the resulting column
flights_assembled.select('features', 'label').show(5, truncate=False)

+-----------------------------------------+-----+
|features                                 |label|
+-----------------------------------------+-----+
|[10.0,10.0,1.0,2.0,0.0,253.0,8.18,51.0]  |1    |
|[11.0,22.0,1.0,2.0,0.0,1188.0,7.17,127.0]|0    |
|[2.0,14.0,5.0,4.0,2.0,3618.0,21.17,365.0]|1    |
|[5.0,25.0,3.0,3.0,5.0,621.0,12.92,85.0]  |1    |
|[3.0,28.0,1.0,4.0,3.0,1732.0,13.33,182.0]|1    |
+-----------------------------------------+-----+
only showing top 5 rows



In [17]:
flights_assembled.count()

258289

In [18]:
# Train/test split
flights_train, flights_test=flights_assembled.randomSplit([0.8,0.2],seed=17)

# Check that training set has around 80% of records
training_ratio=flights_train.count()/flights_assembled.count()
print(training_ratio)

0.7996856234682855


In [19]:
# Build a Decision Tree
from pyspark.ml.classification import DecisionTreeClassifier

# Create a classifier object and fit to the training data
tree=DecisionTreeClassifier()
tree_model=tree.fit(flights_train)

In [20]:
# Create predictions for the testing data and take a look at the predictions
prediction=tree_model.transform(flights_test)
prediction.select('label','prediction','probability').show(54, False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.5638002773925104,0.43619972260748957]|
|1    |0.0       |[0.5638002773925104,0.43619972260748957]|
|0    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|0    |1.0       |[0.4775576223210675,0.5224423776789324] |
|1    |1.0       |[0.47274436090225563,0.5272556390977443]|
|0    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.6278223770314416] |
|1    |1.0       |[0.3721776229685584,0.

In [22]:
# Evaluate the Decision Tree
prediction.groupBy('label','prediction').count().show()

# Calculate the elements of the confusion matrix
TN=prediction.filter('prediction=0 AND label=prediction').count()
TP=prediction.filter('prediction=1 AND label=prediction').count()
FN=prediction.filter('prediction=0 AND label=1').count()
FP=prediction.filter('prediction=1 AND label=0').count()

## Accuracy measures the proportion of correct predictions
accuracy=(TN + TP)/(TN +TP + FN + FP)
print(accuracy)

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 7184|
|    0|       0.0|13915|
|    1|       1.0|18931|
|    0|       1.0|11709|
+-----+----------+-----+

0.6348402558998048


In [23]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

#Calculate precision and recall
precision=TP/(TP+FP)
recall=TP/(TP+FN)
print('precision={:.2f}\nrecall = {:>2f}'.format(precision,recall))

precision=0.62
recall = 0.724909
